# Chapter 4: Multi-Input and Multi-Output Models
**Module 02: Intermediate Deep Learning with PyTorch**  
*Instructor: Michal Oleszak, Machine Learning Engineer*

> Complete PDF-integrated notebook for multi-input datasets, tensor concatenation, multi-output networks, evaluation, and loss weighting.


## Learning Objectives
- Explain why multi-input and multi-output models are useful.
- Implement datasets that return multiple inputs or multiple labels.
- Concatenate model branch features with `torch.cat()`.
- Train two-input models with image and metadata streams.
- Train two-output models with separate task heads.
- Evaluate each output with its own metric.
- Weight and normalize losses when combining tasks.


## 1. Why Multi-Input?
Multi-input models use more than one information source.

Examples from the PDF:
- Using more information.
- Multi-modal models.
- Metric learning.
- Self-supervised learning.

The chapter example uses Omniglot character images plus alphabet metadata.


## 2. Two-Input Dataset
A two-input sample returns:

```text
(image, alphabet_vector, character_label)
```

| Item | Meaning |
|---|---|
| `img` | Grayscale character image |
| `alphabet` | One-hot alphabet vector |
| `label` | Character class |


In [ ]:
from pathlib import Path
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

DATA_DIR = Path("datasets")
OMNI_TRAIN_ZIP = DATA_DIR / "omniglot_train.zip"
OMNI_TEST_ZIP = DATA_DIR / "omniglot_test.zip"
OMNI_TRAIN_DIR = DATA_DIR / "omniglot_train"
OMNI_TEST_DIR = DATA_DIR / "omniglot_test"
for zip_path in [OMNI_TRAIN_ZIP, OMNI_TEST_ZIP]:
    target_dir = DATA_DIR / zip_path.stem
    if zip_path.exists() and not target_dir.exists():
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(DATA_DIR)

print("Omniglot train available:", OMNI_TRAIN_DIR.exists())


In [ ]:
class OmniglotDataset(Dataset):
    def __init__(self, transform, samples):
        self.transform = transform
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, alphabet, label = self.samples[idx]
        img = Image.open(img_path).convert('L')
        img = self.transform(img)
        return img, torch.as_tensor(alphabet), torch.as_tensor(label, dtype=torch.long)

print("Two-input dataset class defined.")


In [ ]:
def build_multi_input_samples(root, max_samples=128):
    image_paths = sorted(Path(root).glob("*/*/*.png"))[:max_samples]
    alphabets = sorted({p.parent.parent.name for p in image_paths})
    alphabet_to_idx = {name: i for i, name in enumerate(alphabets)}
    chars = sorted({p.parent.name for p in image_paths})
    char_to_idx = {name: i for i, name in enumerate(chars)}
    samples = []
    for path in image_paths:
        alphabet_vec = np.zeros(max(1, len(alphabets)), dtype="float32")
        alphabet_vec[alphabet_to_idx[path.parent.parent.name]] = 1.0
        samples.append((str(path), alphabet_vec, char_to_idx[path.parent.name]))
    return samples, alphabet_to_idx, char_to_idx

transform = transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor()])
if OMNI_TRAIN_DIR.exists():
    samples, alphabet_to_idx, char_to_idx = build_multi_input_samples(OMNI_TRAIN_DIR)
else:
    samples, alphabet_to_idx, char_to_idx = [], {}, {}
print("Example sample:", samples[0] if samples else "No extracted samples found")


## 3. Tensor Concatenation
`torch.cat()` combines tensors along a chosen axis.

| Dimension | Meaning in This Chapter |
|---|---|
| `dim=0` | Stack examples vertically |
| `dim=1` | Concatenate features for the same example |


In [ ]:
x = torch.tensor([[1, 2, 3]])
y = torch.tensor([[4, 5, 6]])
print(torch.cat((x, y), dim=0))
print(torch.cat((x, y), dim=1))


## 4. Two-Input Architecture
A two-input architecture processes each input stream separately, concatenates branch outputs, then classifies the combined representation.


In [ ]:
class MultiInputNet(nn.Module):
    def __init__(self, num_alphabets=30, num_classes=964):
        super(MultiInputNet, self).__init__()
        self.image_layer = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.MaxPool2d(kernel_size=2),
            nn.ELU(),
            nn.Flatten(),
            nn.Linear(16 * 32 * 32, 128),
        )
        self.alphabet_layer = nn.Sequential(nn.Linear(num_alphabets, 8), nn.ELU())
        self.classifier = nn.Sequential(nn.Linear(128 + 8, num_classes))

    def forward(self, x_image, x_alphabet):
        x_image = self.image_layer(x_image)
        x_alphabet = self.alphabet_layer(x_alphabet.float())
        x = torch.cat((x_image, x_alphabet), dim=1)
        return self.classifier(x)

num_alphabets = max(1, len(alphabet_to_idx))
num_chars = max(2, len(char_to_idx))
model = MultiInputNet(num_alphabets=num_alphabets, num_classes=num_chars)
print(model(torch.randn(8, 1, 64, 64), torch.randn(8, num_alphabets)).shape)


## 5. Training a Multi-Input Model
Training data contains image, alphabet vector, and label. Pass image and alphabet to the model; compare outputs to labels.


In [ ]:
if samples:
    dataset_train = OmniglotDataset(transform, samples)
    dataloader_train = DataLoader(dataset_train, batch_size=16, shuffle=True)
else:
    dataloader_train = None

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

if dataloader_train is not None:
    model.train()
    for epoch in range(1):
        total_loss = 0.0
        for img, alpha, labels in dataloader_train:
            optimizer.zero_grad()
            outputs = model(img, alpha)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1} | loss={total_loss/len(dataloader_train):.4f}")
else:
    print("No extracted samples found. Training loop is ready for dataloader_train.")


## 6. Why Multi-Output?
Multi-output models predict more than one target.

Examples from the PDF:
- Multi-task learning.
- Multi-label classification.
- Regularization.

The Omniglot example predicts both alphabet and character.


## 7. Two-Output Dataset
A two-output sample returns:

```text
(image, alphabet_label, character_label)
```

The image is the input; the two labels feed separate task losses.


In [ ]:
class OmniglotMultiOutputDataset(Dataset):
    def __init__(self, transform, samples):
        self.transform = transform
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, alphabet, label = self.samples[idx]
        img = Image.open(img_path).convert('L')
        img = self.transform(img)
        return img, torch.as_tensor(alphabet, dtype=torch.long), torch.as_tensor(label, dtype=torch.long)

def build_multi_output_samples(root, max_samples=128):
    image_paths = sorted(Path(root).glob("*/*/*.png"))[:max_samples]
    alphabets = sorted({p.parent.parent.name for p in image_paths})
    alphabet_to_idx = {name: i for i, name in enumerate(alphabets)}
    chars = sorted({p.parent.name for p in image_paths})
    char_to_idx = {name: i for i, name in enumerate(chars)}
    samples = [(str(p), alphabet_to_idx[p.parent.parent.name], char_to_idx[p.parent.name]) for p in image_paths]
    return samples, alphabet_to_idx, char_to_idx

if OMNI_TRAIN_DIR.exists():
    output_samples, alpha_idx, char_idx = build_multi_output_samples(OMNI_TRAIN_DIR)
else:
    output_samples, alpha_idx, char_idx = [], {}, {}
print("Example:", output_samples[0] if output_samples else "No extracted samples found")


## 8. Two-Output Architecture
The image-processing sub-network is shared. Each output task gets its own classifier head.


In [ ]:
class MultiOutputNet(nn.Module):
    def __init__(self, num_alpha, num_char):
        super(MultiOutputNet, self).__init__()
        self.image_layer = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.MaxPool2d(kernel_size=2),
            nn.ELU(),
            nn.Flatten(),
            nn.Linear(16 * 32 * 32, 128),
        )
        self.classifier_alpha = nn.Linear(128, num_alpha)
        self.classifier_char = nn.Linear(128, num_char)

    def forward(self, x):
        x_image = self.image_layer(x)
        output_alpha = self.classifier_alpha(x_image)
        output_char = self.classifier_char(x_image)
        return output_alpha, output_char

num_alpha = max(2, len(alpha_idx))
num_char = max(2, len(char_idx))
net = MultiOutputNet(num_alpha=num_alpha, num_char=num_char)
out_alpha, out_char = net(torch.randn(8, 1, 64, 64))
print(out_alpha.shape, out_char.shape)


## 9. Training a Multi-Output Model
Compute one loss per output, then combine losses before backpropagation.


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.01)

if output_samples:
    dataset_multi_train = OmniglotMultiOutputDataset(transform, output_samples)
    dataloader_multi_train = DataLoader(dataset_multi_train, batch_size=16, shuffle=True)
    net.train()
    for epoch in range(1):
        total_loss = 0.0
        for images, labels_alpha, labels_char in dataloader_multi_train:
            optimizer.zero_grad()
            outputs_alpha, outputs_char = net(images)
            loss_alpha = criterion(outputs_alpha, labels_alpha)
            loss_char = criterion(outputs_char, labels_char)
            loss = loss_alpha + loss_char
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1} | loss={total_loss/len(dataloader_multi_train):.4f}")
else:
    print("No extracted samples found. Multi-output training loop is ready for data.")


## 10. Evaluating Multi-Output Models
Use a separate metric for each output. Alphabet accuracy and character accuracy measure different tasks.


In [ ]:
try:
    from torchmetrics import Accuracy
except ImportError:
    Accuracy = None

if output_samples and Accuracy is not None:
    acc_alpha = Accuracy(task="multiclass", num_classes=num_alpha)
    acc_char = Accuracy(task="multiclass", num_classes=num_char)
    net.eval()
    with torch.no_grad():
        for images, labels_alpha, labels_char in dataloader_multi_train:
            out_alpha, out_char = net(images)
            _, pred_alpha = torch.max(out_alpha, 1)
            _, pred_char = torch.max(out_char, 1)
            acc_alpha(pred_alpha, labels_alpha)
            acc_char(pred_char, labels_char)
    print(f"Alphabet: {acc_alpha.compute()}")
    print(f"Character: {acc_char.compute()}")
else:
    print("Install torchmetrics and provide Omniglot samples to run multi-output evaluation.")


## 11. Loss Weighting
If both tasks are equally important:

```python
loss = loss_alpha + loss_char
```

If character classification is twice as important:

```python
loss = loss_alpha + loss_char * 2
```

Or use weights that sum to one:

```python
loss = 0.33 * loss_alpha + 0.67 * loss_char
```

> **Warning:** Losses must be on the same scale before weighting. Normalize losses when combining very different objectives such as MSE and cross-entropy.


In [ ]:
loss_alpha = torch.tensor(1.2)
loss_char = torch.tensor(2.0)
print("equal:", (loss_alpha + loss_char).item())
print("char twice:", (loss_alpha + loss_char * 2).item())
print("weighted:", (0.33 * loss_alpha + 0.67 * loss_char).item())

loss_price = torch.tensor([10000.0])
loss_quality = torch.tensor([2.0])
loss_price = loss_price / torch.max(loss_price)
loss_quality = loss_quality / torch.max(loss_quality)
loss = 0.7 * loss_price + 0.3 * loss_quality
print("normalized combined loss:", loss.item())


## Course Wrap-Up
You completed the module arc:

1. **Robust neural-network training:** PyTorch OOP, optimizers, vanishing/exploding gradients.
2. **Images and CNNs:** image loading, augmentation, training, and evaluation.
3. **Sequences and recurrent networks:** sequence creation, RNN/LSTM/GRU models, and MSE evaluation.
4. **Flexible architectures:** multi-input models, multi-output models, and loss weighting.

## What to Learn Next
- Transformers
- Self-supervised learning
- Deep Learning for Text with PyTorch
- Deep Learning for Images with PyTorch
